## GX_02 — Preparing the Data (Data Engineering)

**Question:** *Is the data ready for analysis and modelling?*

**Notebook Overview**

GX_01 showed that both files are complete and technically clean, and it listed a few things to deal with: columns we should not use, text that has to be turned into dates and numbers, a PIN code that should not be stored, and patterns (like the 36% of customers with a zero balance) worth turning into their own columns.

Here we act on those findings. Every change is a deliberate decision, explained before the code that makes it.

**Plan**
1. **Clean** both files into *silver* tables.
2. **Build** a *gold* customer table with new, more useful columns and a fixed train / validation / test split.
3. **Prepare** the steps that turn customer data into the numeric format a Spark model needs.
4. **Show why Spark**: run the same steps on 1 million rows and compare the time.

**Deliverables**
| Table | What's in it | Used by |
|---|---|---|
| `silver_churn` | Cleaned customers: only the useful columns, correct types | gold table |
| `silver_transactions` | Cleaned transactions: real dates, numeric coordinates, no PIN code | graph analysis (Phase 6), streaming (Phase 7) |
| `gold_churn_features` | Customers + new columns + train/val/test label | every later notebook |
| `scaletest_churn_x100` | A synthetic 1-million-row copy, **only** for the speed test | nothing else |

**The three layers**

- Bronze: untouched copy of the files (made in GX_01)
- Silver: cleaned data (this notebook)
- Gold: data ready for analysis and models (this notebook)

Every table here is rebuilt from bronze each time the notebook runs. Nothing is edited by hand, so anyone in the team can re-run it and get exactly the same tables.

### **0. Preparing the notebook**

**Approach:** load our shared helper code from `src/` and Spark's machine learning tools.

**Why:** the cleaning and feature rules live in `src/`, not in this notebook, so that later notebooks (and the streaming part in Phase 7) apply *exactly* the same rules to new data. New helper files for this phase:
- `cleaning.py` — the bronze → silver cleaning rules
- `features.py` — the customer groups and the new columns
- `ml.py` — the train/val/test split and the model preparation steps
- `utils.py` — a stopwatch and the synthetic data for the speed test

In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from pyspark.ml import Pipeline
from pyspark.sql import functions as F

from src.cleaning import clean_churn, clean_transactions
from src.config import (
    BRONZE_CHURN, BRONZE_TRANSACTIONS, EXPECTED_ROWS, GOLD_CHURN, SCALE_FACTOR,
    SCALE_TEST_CHURN, SEED, SILVER_CHURN, SILVER_TRANSACTIONS, SPLIT_RATIOS,
)
from src.features import build_gold_features
from src.ingest import write_table
from src.ml import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, add_split_column,
    build_preprocessing_stages, vector_slot_names,
)
from src.quality import quality_report
from src.utils import replicate_with_noise, timer

print("Spark version:", spark.version)

**Result:** Spark is running and all helper code loaded. We start from the bronze tables made in GX_01.

### **1. Cleaning the customer data (`silver_churn`)**

**Approach:** In `bronze_churn`, keep only what the analysis needs, with fixed types.

**Key decisions**

| Column | Decision | Reason |
|---|---|---|
| `RowNumber` | Drop | It is an ID, says nothing about the customer, and a model could even pick up patterns from the file order. |
| `Surname` | Drop | Personal data, and it wouldn't help predict churn. Not storing it is a privacy point. |
| `CustomerId` | Keep, as a key only | Needed to identify customers (e.g. to give the retention team a list), but it is not used as a model input. |
| `Geography`, `Gender` | Trim spaces | Already clean. Trimming protects against a future file where " France" and "France" would become two different countries. |
| `HasCrCard`, `IsActiveMember`, `Exited` | Make them whole numbers (0/1) | They already are. Doing it explicitly makes the rule visible and catches it if a future file sends "Yes"/"No". |
| `_corrupt_record` and tracking columns | Drop | Rows that failed to load would be removed here. GX_01 found 0 such rows, so no customer is lost. |

In [0]:
silver_churn = clean_churn(spark.table(BRONZE_CHURN))
write_table(silver_churn, SILVER_CHURN)

silver_churn = spark.table(SILVER_CHURN)
print(f"{SILVER_CHURN}: {silver_churn.count():,} rows, {len(silver_churn.columns)} columns")
silver_churn.printSchema()

**Result:** Total of 10,000 rows, same as bronze, so no customer was lost. The table went from 17 columns to 12, the 11 customer columns we need plus `CustomerId` as the key. `RowNumber`, `Surname` and the 3 technical columns are gone.

**Approach:** Run the same quality function as in GX_01 on the silver table, and stop the notebook if anything fails.

**Why:** cleaning code can introduce new problems (a wrong type, a column lost, a filter that removes too much). Rechecking data quality after cleaning allows issues to be detected immediately, before they affect subsequent analysis or model development.

In [0]:
silver_churn_quality = quality_report(
    silver_churn,
    dataset="silver_churn",
    expected_rows=EXPECTED_ROWS["churn"],
    key_col="CustomerId",
    ranges={"CreditScore": (300, 850), "Age": (18, 100), "Tenure": (0, 10), "NumOfProducts": (1, 4)},
    allowed_values={
        "Geography": ["France", "Germany", "Spain"],
        "Gender": ["Male", "Female"],
        "HasCrCard": [0, 1],
        "IsActiveMember": [0, 1],
        "Exited": [0, 1],
    },
    rate_cols=["Exited"],
)
display(silver_churn_quality.groupBy("status").count().orderBy("status"))

n_failed = silver_churn_quality.filter("status = 'FAIL'").count()
if n_failed > 0:
    display(silver_churn_quality.filter("status = 'FAIL'"))
    raise ValueError(f"{n_failed} check(s) failed on silver_churn - fix the cleaning step.")
print("silver_churn passed every hard check.")

**Result:** All 28 data quality checks passed, with no failures. One informational entry confirmed that the churn rate remained at 20.4%. The cleaning process changed the table’s structure while preserving its underlying data.

### 2. Cleaning the transaction data (`silver_transactions`)

**Approach:** turn the text columns that GX_01 kept "as is" into usable values.

**Key decisions:**

| Column | Decision | Reason |
|---|---|---|
| `timestamp_raw` (text) | Turn into a real date/time, `event_time` | Converting text values such as "2025-01-17 10:14:00" to timestamps enables reliable time-based sorting, grouping, and calculations required in Phase 7 (streaming). |
| `geolocation_raw` (text) | Split into two numbers, `latitude` and `longitude` | Numeric types allow values to be compared and mapped correctly. |
| Hemisphere letters (N/S/E/W) | Ignore them, trust the numbers  sign |GX_01 showed that all rows use “W”, including positive longitudes such as Paris (2.35). Since the direction labels conflict with the coordinates, we use the numeric values. |
| `pin_code` | Drop | A security code has no analytical use, and storing it is a privacy and security risk. |

In [0]:
silver_transactions = clean_transactions(spark.table(BRONZE_TRANSACTIONS))
write_table(silver_transactions, SILVER_TRANSACTIONS)

silver_transactions = spark.table(SILVER_TRANSACTIONS)
print(f"{SILVER_TRANSACTIONS}: {silver_transactions.count():,} rows, {len(silver_transactions.columns)} columns")
silver_transactions.printSchema()

**Result:** All 1000 rows were retained.`event_time` is now a timestamp,`latitude` and `longitude` are numeric, and `pin_code` has been removed.

**Approach:** Check that the conversions succeeded by counting null values, which may indicate dates or coordinates that could not be converted.

In [0]:
silver_transactions.createOrReplaceTempView("silver_transactions")

display(spark.sql('''
    SELECT
        COUNT(*)                                          AS rows,
        COUNT_IF(event_time IS NULL)                      AS unreadable_times,
        COUNT_IF(latitude IS NULL OR longitude IS NULL)   AS unreadable_locations,
        MIN(event_time)                                   AS first_transaction,
        MAX(event_time)                                   AS last_transaction,
        COUNT(DISTINCT event_time)                        AS distinct_minutes,
        COUNT(DISTINCT latitude)                          AS distinct_latitudes,
        COUNT(DISTINCT longitude)                         AS distinct_longitudes,
        COUNT_IF(longitude < 0)                           AS west_of_greenwich
    FROM silver_transactions
'''))

**Result:**
- All 1,000 rows were converted successfully, with no unreadable dates or coordinates.
- Timestamps range from 10:01 to 11:00 on 17 January 2025, covering 60 distinct minutes. Since all timestamps are already recorded at minute precision, a separate minute column is unnecessary.
- The data contains 6 distinct latitudes and 6 distinct longitudes, consistent with GX_01. Based on the numeric coordinates, 310 transactions are west of Greenwich and 690 are east. However, inconsistencies remain: London’s longitude is recorded as +0.1278, placing it east of Greenwich rather than west.

Cleaning corrected the data format, but not the underlying inaccuracies. The coordinates were retained for completeness, but will not be used to draw geographical conclusions.

**Approach:** Compare the original location text in Bronze with the parsed coordinates in Silver for a few transactions to visually verify the conversion.

In [0]:
display(
    spark.table(BRONZE_TRANSACTIONS).select("transaction_id", "geolocation_raw")
    .join(silver_transactions.select("transaction_id", "latitude", "longitude"), "transaction_id")
    .orderBy("transaction_id")
    .limit(5)
)

**Result:** The conversion preserves the sign of each numeric coordinate and ignores the direction letters.
- For TXN1026668467, "34.0522 N, 2.3522 W" becomes latitude 34.0522 and longitude +2.3522. The positive longitude places the point east of Greenwich, despite the original W label.
- For TXN1038301181, "-74.006 W" becomes −74.006. Here, the negative sign and the direction label agree.
- Coordinate inconsistencies remain. TXN1026668467 combines Los Angeles’s latitude with Paris’s longitude, producing a point in northern Algeria. This illustrates why successful conversion does not guarantee accurate location data.

### 3. Building the gold customer table (`gold_churn_features`)

**Approach:** Add features that describe each customer and assign each customer to a fixed training, validation, or test set.

**Why:** A model learns from the columns provided. Additional features can make relevant patterns easier to identify. Each new feature is based on findings from GX_01:

| New column | What it is | Why |
|---|---|---|
| `AgeGroup` | 18–30 / 31–40 / 41–50 / 51–60 / 60+ | Churn changes with age, and groups are easy to explain to management |
| `TenureGroup` | 0–2 / 3–6 / 7–10 years | Easy grouping for charts |
| `BalanceGroup`, `CreditScoreGroup` | Bands of balance and credit score | Same groups in the charts (GX_03) and the segments (Phase 5) |
| `ProductsFlag` | 1 / 2 / 3+ products | Only 326 customers have 3 or 4 products, so they are grouped together |
| `ZeroBalance` | 1 if the balance is exactly 0 | 36% of customers have an empty account; they may behave differently |
| `BalanceToSalary` | Balance divided by yearly salary | How much of their income a customer keeps with the bank |
| `LogBalanceToSalary` | A compressed version of the same ratio | Explained in section 3.1 |
| `Engagement` | 0 to 3: active member + has a credit card + has 2 or more products | One number for "how tied is this customer to the bank" |

Group boundaries are defined once in `src/features.py`, ensuring that the slides, model, and customer segments use the same bands.

**Why a fixed train / validation / test label:**
- Training (70%) — used to train the model.
- Validation (15%) — used to compare models and select settings.
- Test (15%) — reserved for the final evaluation on customers not used during training or model selection.

The split is assigned once and stored in the table. This ensures that modelling, segmentation, and graph analysis use consistent customer groups. Training notebooks must explicitly exclude test customers.

The split is stratified, churners and non-churners are divided separately to preserve approximately the same churn rate (20.4%) in each set. It is also reproducible, customers are ordered using a fixed hash of their ID rather than a random draw, producing the same assignments across runs and environments when the data and splitting logic remain unchanged.

In [0]:
gold = (
    build_gold_features(silver_churn)
    .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
)
write_table(gold, GOLD_CHURN)

gold = spark.table(GOLD_CHURN).cache()   # small table, read several times below
gold.createOrReplaceTempView("gold")
print(f"{GOLD_CHURN}: {gold.count():,} rows, {len(gold.columns)} columns")

display(gold.select(
    "CustomerId", "Age", "AgeGroup", "Balance", "ZeroBalance", "NumOfProducts", "ProductsFlag",
    "IsActiveMember", "HasCrCard", "Engagement", "BalanceToSalary", "split", "Exited",
).limit(5))

**Result:** The table contains 10,000 rows and 22 columns: the 12 Silver columns, 9 new features, and the split label.
The preview allows the new features to be checked manually. For example, the second customer is aged 66, has two products, is active, and has no credit card. Their Engagement score is therefore 2: 1 for being active, 0 for having no credit card, and 1 for having at least two products. Their AgeGroup is "60+".

### 3. 1 Do the new columns tell us anything?

**Approach:** Compare the churn rate across the values of two new columns, using Spark SQL on the gold table.

**Why:** A new column is only worth keeping if it separates customers who leave from customers who stay. If the churn rate is the same for every value, the column adds nothing.

In [0]:
display(spark.sql('''
    SELECT 'ZeroBalance' AS feature, CAST(ZeroBalance AS STRING) AS value,
           COUNT(*) AS customers, ROUND(AVG(Exited) * 100, 1) AS churn_rate_pct
    FROM gold GROUP BY ZeroBalance
    UNION ALL
    SELECT 'Engagement', CAST(Engagement AS STRING),
           COUNT(*), ROUND(AVG(Exited) * 100, 1)
    FROM gold GROUP BY Engagement
    ORDER BY feature, value
'''))

**Result:** Both columns clearly separate the two groups.
- ZeroBalance: Customers with a zero balance have a churn rate of 13.8%, compared with 24.1% for those with a positive balance. No German customer has a zero balance, so this pattern may partly reflect differences between countries. This will be examined in later analysis.
- Engagement: Churn decreases as engagement rises: 34.5% at score 0, 29.3% at 1, 15.9% at 2, and 9.1% at 3. Customers with the highest score have roughly one-quarter the churn rate of those with the lowest score.

Both features are retained.

**Approach:** Analyze how the balance-to-salary ratio is spread out.

**Why:** GX_01_ingestion identified 59 customers with an annual salary below €1,000, with a minimum of €11.58. Dividing the account balance by such small values can produce extremely high ratios. Their range will be examined before using this feature in the model.

In [0]:
display(gold.select("BalanceToSalary", "LogBalanceToSalary").summary("min", "25%", "50%", "75%", "max", "mean", "stddev"))

**Result:** The ratio is highly skewed. Half of customers have values below 0.75, and 75% are below 1.51, but the maximum reaches 10,615, driven by unusually low reported salaries. These extreme values raise the mean to 3.9, around five times the median, and the standard deviation to 108.

**Decision:** The model uses LogBalanceToSalary, calculated as log(1 + ratio), to reduce the influence of extreme values. This transformation preserves the ordering while reducing the maximum to 9.3 and the standard deviation to 0.77. The original ratio is retained for charts because it is easier to interpret.

### 3.2 Checking the train / validation / test split

**Approach:** Count the customers and the churn rate in each part.

**Why:** Different churn rates across the sets could affect model evaluation, making performance appear better or worse because of the split rather than what the model learned.

In [0]:
display(spark.sql('''
    SELECT split,
           COUNT(*)                        AS customers,
           SUM(Exited)                     AS churned,
           ROUND(AVG(Exited) * 100, 2)     AS churn_rate_pct,
           ROUND(COUNT(*) / 10000 * 100, 1) AS share_pct
    FROM gold
    GROUP BY split
    ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
'''))

**Result:** Split works as expected.

| Part | Customers | Churned | Churn rate |
|---|---|---|---|
| Train | 7,000 | 1,426 | 20.37% |
| Validation | 1,499 | 305 | 20.35% |
| Test | 1,501 | 306 | 20.39% |

The training, validation, and test sets follow a 70/15/15 split, each with a churn rate of 20.4%, matching the overall customer base.

### 4. Model preparation steps (Spark ML Pipeline)

**Approach:** Define and test the steps that transform each customer row into the feature vector required by a Spark model.

**Why:** Spark models require numeric inputs, and some benefit from features on similar scales. The following four steps are combined in a Pipeline, ensuring consistent preprocessing.

| Step | What it does | Example |
|---|---|---|
| 1. StringIndexer | Converts text categories into numbers | France → 0, Germany → 1, Spain → 2 |
| 2. OneHotEncoder | Converts that number into separate 0/1 columns | Germany → [0, 1, 0]. Otherwise a model would read "Spain = 2" as "twice Germany". |
| 3. VectorAssembler | Combine all inputs into one list of numbers | [0, 1, 0, 1, 0, 619, 42, ...] |
| 4. StandardScaler | Convert every input into the same scale (average 0, spread 1) | Without it, a balance of 100000 would dominate an age of 42 |

**Input Features**
- Text: `Geography`, `Gender`
- Numbers: credit score, age, tenure, balance, number of products, credit card, active member, salary, `ZeroBalance`, `LogBalanceToSalary`, `Engagement`
- Not used: `CustomerId`, the group labels (model sees the exact values instead), and `Exited`(target).

**Rule: Fit the preprocessing pipeline on the training set only, then apply it to all three sets. The scaler learns scaling statistics, while the indexer learns the categories present in the training data. Including validation or test customers would introduce data leakage and could make the final evaluation overly optimistic.

In Phase 4, the model is will be added as the fifth and final step in the same pipeline.

In [0]:
preprocessing = Pipeline(stages=build_preprocessing_stages(CATEGORICAL_FEATURES, NUMERIC_FEATURES))

train = gold.filter("split = 'train'")
preprocessing_model = preprocessing.fit(train)        # learns ONLY from the 7,000 training customers
prepared = preprocessing_model.transform(gold)         # ...then applies the same rules to all 10,000

slot_names = vector_slot_names(prepared)
print(f"Each customer becomes a list of {len(slot_names)} numbers:")
for i, name in enumerate(slot_names):
    print(f"  {i:2d}  {name}")

**Result:** Each customer is now represented by a feature vector of 16 numbers: 3 for country, 2 for gender, and 11 numeric features.

The country indexer reserves an additional category for unknown values, allowing new countries to be processed without failing. The encoder drops this extra category by default, leaving a separate column for France, Germany, and Spain. An unknown country is represented by zeros in all three country columns.

**Approach:** Inspect what the preprocessing steps learned from the training set and review one customer’s feature vector.

In [0]:
indexer_model = preprocessing_model.stages[0]
scaler_model = preprocessing_model.stages[-1]

print("Categories learned from train:", indexer_model.labelsArray)

learned = dict(zip(slot_names, scaler_model.mean.toArray()))
print("Averages learned from train:")
for col in ["CreditScore", "Age", "Balance", "EstimatedSalary"]:
    print(f"  {col:16s} {learned[col]:>12,.1f}")

display(prepared.select("CustomerId", "Geography", "Gender", "Age", "Balance", "features").limit(3))

**Result:**
- The indexer identified 3 countries and 2 genders, ordered by frequency: France, Germany, Spain; Male, Female.
- Using only the training set, the scaler calculated mean values of approximately 651 for credit score, 39 for age, €77,000 for balance, and €100,000 for salary.
- Standardised features are centred on the training mean: positive values indicate above-average values, and negative values indicate below-average values. For example, the 66-year-old customer has a scaled age of +2.6, while the 35-year-old has −0.4.

The prepared data is not saved as a separate table. In Phase 4, preprocessing is rebuilt using the same function and combined with the model in a single pipeline, so both are saved and reused together.

## 5. Why Spark? A scalability test

**Client's question:** *"We have 10,000 customers. Why not just use Excel or a laptop script?"*

**Our answer:** This prototype is designed for a bank with millions of customers and years of transaction history. To assess its scalability, we run **the same processing steps** on a larger dataset and measure the execution time.


**Approach:**
1. Create a synthetic customer dataset that is 100 times larger than the original (1,000,000 rows). Each copied record receives a new ID, with balance and salary adjusted by up to ±1%.
2. Run the same workflow on both datasets: cleaning → Gold features → data split → fit preprocessing on the training set → transform all customers.
3. Compare execution times.

The synthetic data is used only for performance testing. It is excluded from all customer analysis, modelling results, and business conclusions.


**How Spark handles larger datasets:**

Spark divides data into partitions and processes them in parallel. A larger cluster can process more partitions simultaneously without changing the workflow code, although performance gains depend on the workload and available resources.
This test evaluates whether the same code can process 100 times more data and measures how execution time changes.

In [0]:
def prepare_customers(bronze_customers):
    # The full Phase 2 chain, from raw customers to model-ready customers.
    gold_df = (
        build_gold_features(clean_churn(bronze_customers))
        .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
    )
    model = Pipeline(stages=build_preprocessing_stages()).fit(gold_df.filter("split = 'train'"))
    return model.transform(gold_df)


bronze_customers = spark.table(BRONZE_CHURN)
big_bronze_customers = replicate_with_noise(bronze_customers, SCALE_FACTOR, SEED)
print(f"Original: {bronze_customers.count():,} rows   Synthetic copy: {big_bronze_customers.count():,} rows")

**Result:** The synthetic copy has 1000000 rows, 100 times the original 10000.

**Approach:** Measure the full workflow on both datasets. Writing the output to a no-operation sink (noop) forces Spark to execute the workflow without storing the results, allowing us to measure processing time without the cost of saving the output.

**Importance:** Spark uses lazy evaluation: transformations such as clean_churn(...) define the work but do not execute it. An action is needed to trigger processing; otherwise, we would measure only the time taken to define the workflow.

In [0]:
timings = {}

with timer("10,000 customers", timings):
    prepare_customers(bronze_customers).write.format("noop").mode("overwrite").save()

with timer("1,000,000 customers (synthetic)", timings):
    prepare_customers(big_bronze_customers).write.format("noop").mode("overwrite").save()

display(spark.createDataFrame(list(timings.items()), ["run", "seconds"]))

small, big = timings.values()
print(f"Data grew {SCALE_FACTOR}x; processing time grew {big / small:.1f}x "
      f"({small:.1f}s -> {big:.1f}s), with no change to the code.")

**Result:** In this run, processing 100 times more data took approximately 2.4 times longer, increasing from 3.6 to 8.7 seconds.
- The same code processed both datasets without modification, with no memory errors or row limits encountered.
- Execution time increased much less than data volume. Fixed costs, such as job scheduling and query planning, likely account for part of this difference, as they are spread across more rows in the larger dataset.
- Execution times vary between runs, so these results provide an indicative comparison rather than a performance guarantee.

**Implications for the client:** The test demonstrates that the pipeline can process one million synthetic customer records without code changes. This supports its scalability, although production readiness requires further testing with realistic data volumes, transaction history, and workloads.v

### 5.1 Storing big data smartly: partitioning

**Approach:** Save the one-million-row Gold table partitioned by country using partitionBy("Geography"). Query German customers only, then inspect the execution plan with explain().

**Why:** Queries may focus on a single country or region. Partitioning by country allows Spark to skip irrelevant partitions rather than read the entire dataset. The execution plan shows whether Spark applies this optimisation, known as partition pruning.

In [0]:
big_gold = (
    build_gold_features(clean_churn(big_bronze_customers))
    .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
)
write_table(big_gold, SCALE_TEST_CHURN, partition_by=["Geography"])

germany_only = spark.table(SCALE_TEST_CHURN).filter(F.col("Geography") == "Germany")
print(f"German customers in the synthetic table: {germany_only.count():,}")
germany_only.select("CustomerId", "Age", "Balance", "Exited").explain()

**Result:** The query returns 250,900 German customer records, corresponding to the 2,509 original German customers replicated 100 times.
The execution plan includes PartitionFilters: [... (Geography = Germany)], confirming that Spark selects only the Germany partition and skips France and Spain. This excludes approximately 75% of the rows from the scan.
The ReadSchema also lists only the four requested columns, showing that Spark avoids reading the remaining columns.
Partition and column pruning reduce the amount of data read, which can improve query performance and lower processing costs as data volumes grow.

### 6. Summary

**What we did**

- Silver customers (`silver_churn`): Removed row number and surname, retained customer ID as a key, and corrected data types. All 10,000 customers were retained, and all quality checks passed after cleaning.
- Silver transactions (`silver_transactions`): Converted dates to timestamps and coordinates to numeric values, preserving their signs rather than the inconsistent direction labels. Removed PIN codes and retained all 1,000 transactions, with no conversion failures.
- Gold customers (`gold_churn_features`): Added 9 features and a fixed, stratified, reproducible split: 7,000 training, 1,499 validation, and 1,501 test customers, each with a churn rate of approximately 20.4%.
- Preprocessing pipelin Defined four steps that transform each customer into a **16-value feature vect, with preprocessing fitted on the training set only.
- Scalability test: Processeone million synthetic records using the same workflow without code changes. Country partitioning allowed Spark to exclude approximately 75% of rows from a Germany-only scan.

**What we learned**

- Higher engagement is associated with lower churn: Rates fall from 34.5% at engagement score 0 to 9.1% at score 3.
- Zero-balance customers have lower observed churn: Their churn rate is 13.8%, compared with 24.1% for customers with a positive balance. Country differences may partly explain this pattern.
- Very low reported salaries create extreme ratios: The balance-to-salary ratio reaches 10615, so the model uses a logarithmic transformation to reduce the influence of extreme values.
- Cleaning improves format, not underlying accuracy: Transaction coordinates remain inconsistent, the data covers only one hour, and transactions cannot be linked to customers.

**Decisions for later notebooks**

- Use `gold_churn_features` as the input for subsequent customer analysis and modelling.
- Respect the `split` assignments: train on `train`, compare models and select settings on `val`, and reserve `test` for the final evaluation.
- Build model pipelines using `build_preprocessing_stages()`, followed by the model, so preprocessing and prediction are saved and reused together.

**Next: GX_03 — Exploring the data with Spark SQL**

The next notebook uses the Gold table to investigate the client’s first business question: **Which customers are more likely to leave, and how does churn vary by country?**

- Compare churn by country, gender, age group, product count, and activity using Spark SQL and the DataFrame API.
- Examine combinations such as country × zero balance and age × activity.
- Use correlations and chi-square tests to assess associations and the evidence for observed differences.
- Present 3–4 key findings, each supported by a chart, for management reporting and further investigation during modelling in Phase 4.